# Đánh giá ablation C0–C3 trên Colab

Chọn **Runtime → Change runtime type → T4 GPU** trước khi chạy. Notebook giữ dữ liệu và kết quả trong Drive nên có thể tiếp tục sau khi Colab ngắt.

Chuẩn bị đúng cấu trúc sau:

```text
MyDrive/Advanced_colab/
├── integrated_test/
│   ├── mix/                 # 120 WAV mixture
│   ├── target/              # 120 WAV target tham chiếu
│   ├── interferer/
│   ├── noise/
│   └── metadata.csv
├── checkpoints/
│   ├── separation/finetuned/best.pt
│   └── enhancement/finetuned/DTLN_vi_snr5_lr1e-4_run01.weights.h5
└── outputs/                 # notebook tự tạo
```

Hãy upload **nguyên thư mục** `D:/Codes/AdvancedML/data/processed/integrated_test` vào `MyDrive/Advanced_colab/integrated_test`. Hai checkpoint local cũng phải được upload đúng hai đường dẫn trên.

In [40]:
# 1. Mount Drive và khai báo đường dẫn
from google.colab import drive
from pathlib import Path
import os
import subprocess
import sys

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/Advanced_colab')
DATA_ROOT = DRIVE_ROOT / 'integrated_test'
METADATA = DATA_ROOT / 'metadata.csv'
SEP_CHECKPOINT = DRIVE_ROOT / 'checkpoints/separation/finetuned/best.pt'
DTLN_CHECKPOINT = DRIVE_ROOT / 'checkpoints/enhancement/finetuned/dtln_finetuned_best.keras'
OUTPUT_DIR = DRIVE_ROOT / 'outputs/integrated_ablation2'
LOCAL_PROJECT = Path('/content/vietnamese-voice-qa')
DTLN_SOURCE = Path('/content/DTLN')
REPO_URL = 'https://github.com/thieuphong220203/vietnamese-voice-qa.git'

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(DRIVE_ROOT / 'model_cache/huggingface')
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
os.environ['TF_USE_LEGACY_KERAS'] = '1'
print('Data:', DATA_ROOT)
print('Output:', OUTPUT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Data: /content/drive/MyDrive/Advanced_colab/integrated_test
Output: /content/drive/MyDrive/Advanced_colab/outputs/integrated_ablation2


In [41]:
# 2. Clone/pull code đã push lên GitHub
if not LOCAL_PROJECT.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(LOCAL_PROJECT)], check=True)
elif (LOCAL_PROJECT / '.git').is_dir():
    subprocess.run(['git', '-C', str(LOCAL_PROJECT), 'pull', '--ff-only'], check=True)
else:
    raise RuntimeError(f'{LOCAL_PROJECT} tồn tại nhưng không phải Git repository')

assert (LOCAL_PROJECT / 'scripts/evaluate_integrated_ablation.py').is_file(), (
    'Chưa thấy evaluator. Hãy commit và push file scripts/evaluate_integrated_ablation.py.'
)
%cd /content/vietnamese-voice-qa
!git pull

/content/vietnamese-voice-qa
Already up to date.


In [4]:
# # 3. Cài dependency. Chạy ô này trước khi import numpy/tensorflow/torch.
# %pip install -q PyYAML soundfile speechbrain transformers torchaudio
# %pip install -q -r requirements-dtln.txt

In [ ]:
# import os
# os.kill(os.getpid(), 9)

In [5]:
# 4. Kiểm tra GPU và phiên bản thư viện
import numpy as np
import torch
import tensorflow as tf

print("NumPy:", np.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ")
print("TensorFlow:", tf.__version__)
print("TensorFlow GPUs:", tf.config.list_physical_devices("GPU"))

if not torch.cuda.is_available():
    raise RuntimeError("Chưa bật GPU: Runtime → Change runtime type → T4 GPU")

NumPy: 2.1.3
PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
TensorFlow: 2.21.0
TensorFlow GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [42]:
# 5. Kiểm tra đủ 120 mẫu, 20 tổ hợp và hai checkpoint
import csv
from collections import Counter

for required in (METADATA, SEP_CHECKPOINT, DTLN_CHECKPOINT):
    if not required.is_file():
        raise FileNotFoundError(f'Thiếu file trên Drive: {required}')

with METADATA.open(encoding='utf-8-sig', newline='') as handle:
    rows = list(csv.DictReader(handle))
distribution = Counter((float(row['snr_db']), float(row['sir_db'])) for row in rows)
assert len(rows) == 120, f'Cần 120 mẫu, nhận {len(rows)}'
assert len(distribution) == 20, f'Cần 20 tổ hợp SNR×SIR, nhận {len(distribution)}'
assert set(distribution.values()) == {6}, distribution
assert {sir for _, sir in distribution} == {1.0, 2.0, 3.0, 5.0}
assert {snr for snr, _ in distribution} == {-5.0, 0.0, 5.0, 10.0, 15.0}
print('Dữ liệu hợp lệ: 120 mẫu, mỗi tổ hợp có 6 mẫu')
print('SepFormer checkpoint:', SEP_CHECKPOINT, SEP_CHECKPOINT.stat().st_size / 1024**2, 'MB')
print('DTLN checkpoint:', DTLN_CHECKPOINT, DTLN_CHECKPOINT.stat().st_size / 1024**2, 'MB')

Dữ liệu hợp lệ: 120 mẫu, mỗi tổ hợp có 6 mẫu
SepFormer checkpoint: /content/drive/MyDrive/Advanced_colab/checkpoints/separation/finetuned/best.pt 107.86841583251953 MB
DTLN checkpoint: /content/drive/MyDrive/Advanced_colab/checkpoints/enhancement/finetuned/dtln_finetuned_best.keras 3.8438339233398438 MB


In [20]:
# for path in (SEP_CHECKPOINT, DTLN_CHECKPOINT, METADATA):
#     print(path, '→', path.is_file())

/content/drive/MyDrive/Advanced_colab/checkpoints/separation/finetuned/best.pt → True
/content/drive/MyDrive/Advanced_colab/checkpoints/enhancement/finetuned/dtln_finetuned_best.weights.h5 → True
/content/drive/MyDrive/Advanced_colab/integrated_test/metadata.csv → True


In [43]:
# 6. Chỉ kiểm tra input/checkpoint trước khi tải model
common_command = [
    sys.executable, 'scripts/evaluate_integrated_ablation.py',
    '--metadata', str(METADATA),
    '--output-dir', str(OUTPUT_DIR),
    '--separation-checkpoint', str(SEP_CHECKPOINT),
    '--enhancement-checkpoint', str(DTLN_CHECKPOINT),
    '--dtln-source-dir', str(DTLN_SOURCE),
    '--sepformer-savedir', str(DRIVE_ROOT / 'model_cache/sepformer-whamr16k'),
    '--torch-device', 'cuda',
    '--dtln-device', 'auto',
    '--dtype', 'float16',
]
subprocess.run(common_command + ['--validate-only'], cwd=LOCAL_PROJECT, check=True)

CompletedProcess(args=['/usr/bin/python3', 'scripts/evaluate_integrated_ablation.py', '--metadata', '/content/drive/MyDrive/Advanced_colab/integrated_test/metadata.csv', '--output-dir', '/content/drive/MyDrive/Advanced_colab/outputs/integrated_ablation2', '--separation-checkpoint', '/content/drive/MyDrive/Advanced_colab/checkpoints/separation/finetuned/best.pt', '--enhancement-checkpoint', '/content/drive/MyDrive/Advanced_colab/checkpoints/enhancement/finetuned/dtln_finetuned_best.keras', '--dtln-source-dir', '/content/DTLN', '--sepformer-savedir', '/content/drive/MyDrive/Advanced_colab/model_cache/sepformer-whamr16k', '--torch-device', 'cuda', '--dtln-device', 'auto', '--dtype', 'float16', '--validate-only'], returncode=0)

In [47]:
# result = subprocess.run(
#     common_command + ["--resume"],
#     cwd=LOCAL_PROJECT,
#     text=True,
#     stdout=subprocess.PIPE,
#     stderr=subprocess.STDOUT,
# )

# print(result.stdout[-12000:])
# print("\nExit code:", result.returncode)

In [44]:
import os
from pathlib import Path

# Cache model ở ổ local Colab vì Google Drive không hỗ trợ symbolic link.
os.environ["HF_HOME"] = "/content/hf_cache"

local_sepformer_cache = Path(
    "/content/model_cache/sepformer-whamr16k"
)
local_sepformer_cache.mkdir(parents=True, exist_ok=True)

# Thay đường dẫn cache cũ trong common_command.
savedir_index = common_command.index("--sepformer-savedir") + 1
common_command[savedir_index] = str(local_sepformer_cache)

print(
    "SepFormer cache:",
    common_command[savedir_index],
)

SepFormer cache: /content/model_cache/sepformer-whamr16k


In [45]:
# 7. Chạy C0–C3. Có thể chạy lại chính ô này sau khi Colab ngắt.
# --resume bỏ qua các (sample, configuration) đã ghi thành công trên Drive.
subprocess.run(common_command + ['--resume'], cwd=LOCAL_PROJECT, check=True)

CompletedProcess(args=['/usr/bin/python3', 'scripts/evaluate_integrated_ablation.py', '--metadata', '/content/drive/MyDrive/Advanced_colab/integrated_test/metadata.csv', '--output-dir', '/content/drive/MyDrive/Advanced_colab/outputs/integrated_ablation2', '--separation-checkpoint', '/content/drive/MyDrive/Advanced_colab/checkpoints/separation/finetuned/best.pt', '--enhancement-checkpoint', '/content/drive/MyDrive/Advanced_colab/checkpoints/enhancement/finetuned/dtln_finetuned_best.keras', '--dtln-source-dir', '/content/DTLN', '--sepformer-savedir', '/content/model_cache/sepformer-whamr16k', '--torch-device', 'cuda', '--dtln-device', 'auto', '--dtype', 'float16', '--resume'], returncode=0)

In [46]:
# 8. Hiển thị bảng kết quả để chép vào bài báo
import pandas as pd

summary_path = OUTPUT_DIR / 'integrated_ablation_summary.csv'
details_path = OUTPUT_DIR / 'integrated_ablation_details.csv'
condition_path = OUTPUT_DIR / 'integrated_ablation_by_condition.csv'
run_info_path = OUTPUT_DIR / 'run_info.json'
if not summary_path.is_file():
    raise FileNotFoundError('Chưa có summary; ô đánh giá chưa chạy xong toàn bộ 120 mẫu')
summary = pd.read_csv(summary_path)
display(summary[[
    'config', 'description', 'sepformer_finetuned', 'dtln_finetuned',
    'phowhisper_small_wer_percent', 'phowhisper_small_cer_percent',
    'preprocessing_rtf', 'mean_latency_s',
]])
print('Summary:', summary_path)
print('Chi tiết 480 lượt:', details_path)
print('Phân tích theo SNR×SIR:', condition_path)
print('Phần cứng/checkpoint:', run_info_path)

,config,description,sepformer_finetuned,dtln_finetuned,phowhisper_small_wer_percent,phowhisper_small_cer_percent,preprocessing_rtf,mean_latency_s
0,C0,Không tiền xử lý,Tắt,Tắt,82.5513,68.2749,0.000000,0.966986
1,C1,Chỉ tách tiếng nói,Bật,Tắt,38.3816,30.6484,0.126914,2.693709
2,C2,Chỉ tăng cường tiếng nói,Tắt,Bật,90.7277,76.8963,0.014847,1.225266
3,C3,Kết hợp hai mô-đun,Bật,Bật,41.9219,33.7327,0.139272,2.829872


Summary: /content/drive/MyDrive/Advanced_colab/outputs/integrated_ablation2/integrated_ablation_summary.csv
Chi tiết 480 lượt: /content/drive/MyDrive/Advanced_colab/outputs/integrated_ablation2/integrated_ablation_details.csv
Phân tích theo SNR×SIR: /content/drive/MyDrive/Advanced_colab/outputs/integrated_ablation2/integrated_ablation_by_condition.csv
Phần cứng/checkpoint: /content/drive/MyDrive/Advanced_colab/outputs/integrated_ablation2/run_info.json


In [18]:
import json
import pandas as pd

summary_path = OUTPUT_DIR / "integrated_ablation_summary.csv"
run_info_path = OUTPUT_DIR / "run_info.json"

summary = pd.read_csv(summary_path)

display(summary[
    [
        "config",
        "phowhisper_small_wer_percent",
        "phowhisper_small_cer_percent",
        "preprocessing_rtf",
        "mean_latency_s",
        "target_selection_accuracy_percent",
    ]
])

with run_info_path.open(encoding="utf-8") as handle:
    run_info = json.load(handle)

print("SepFormer loaded:", run_info["sepformer_loaded_checkpoint"])
print("DTLN loaded:", run_info["dtln_loaded_checkpoint"])
print("GPU:", run_info["gpu"])

,config,phowhisper_small_wer_percent,phowhisper_small_cer_percent,preprocessing_rtf,mean_latency_s,target_selection_accuracy_percent
0,C0,82.5513,68.2749,0.000000,0.987269,NaN
1,C1,38.3816,30.6484,0.128754,2.725373,89.1667
2,C2,95.1672,78.8472,0.017253,1.274936,NaN
3,C3,42.8772,34.2103,0.138581,2.859784,89.1667


SepFormer loaded: /content/drive/MyDrive/Advanced_colab/checkpoints/separation/finetuned/best.pt
DTLN loaded: /content/drive/MyDrive/Advanced_colab/checkpoints/enhancement/finetuned/DTLN_vi_snr5_lr1e-4_run01.weights.h5
GPU: Tesla T4


## File đầu ra

Kết quả được giữ tại `MyDrive/Advanced_colab/outputs/integrated_ablation/`:

- `integrated_ablation_details.csv`: transcript, lỗi và thời gian của từng mẫu × cấu hình (480 dòng).
- `integrated_ablation_summary.csv`: bốn hàng C0–C3 để điền vào Bảng 3.
- `integrated_ablation_by_condition.csv`: kết quả riêng cho từng tổ hợp SNR × SIR.
- `run_info.json`: GPU, phiên bản model và checkpoint đã dùng.
- `run_spec.json`: dấu vân tay dữ liệu/checkpoint để bảo vệ chế độ resume.